# Warehouse_017 — Single-Camera Tracking Pipeline

Produces `Camera.json` + `fixed_Camera.json` for all 8 cameras and saves them to Google Drive.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add your HuggingFace token to Colab Secrets (left panel → 🔑 key icon):
   - Name: `HF_TOKEN`  Value: your token from huggingface.co/settings/tokens
3. Accept dataset terms at: huggingface.co/datasets/nvidia/PhysicalAI-SmartSpaces

Run cells **top to bottom**. Outputs save automatically to Drive.

---
## Step 0 — Mount Drive + Clone Repo

In [ ]:
import os, sys, subprocess, shutil

SCENE   = 'Warehouse_017'
DATASET = 'Test'
CAMERAS = ['Camera', 'Camera_01', 'Camera_02', 'Camera_03',
           'Camera_04', 'Camera_05', 'Camera_06', 'Camera_07']

REPO  = '/content/repo'
DRIVE = '/content/drive/MyDrive/AIC25'

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

for d in ['models', 'outputs/Detection', 'outputs/EmbedFeature', 'outputs/Tracking',
          f'datasets/{DATASET}/{SCENE}']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)

if not os.path.exists(REPO):
    os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
    print('Repo cloned.')
else:
    os.system(f'git -C {REPO} pull --quiet')
    print('Repo updated.')

os.chdir(REPO)
print(f'REPO  : {REPO}')
print(f'DRIVE : {DRIVE}')
print(f'Scene : {SCENE} ({DATASET})')

---
## Step 1 — Install dependencies

In [ ]:
pkgs = ['thop', 'loguru', 'lap', 'motmetrics', 'filterpy',
        'easydict', 'yacs', 'termcolor', 'prettytable',
        'tabulate', 'ninja', 'cython_bbox', 'pycocotools', 'huggingface_hub']
for p in pkgs:
    r = subprocess.run(['pip', 'install', '-q', p], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  {p}: FAILED — {r.stderr.strip()[-80:]}')

if subprocess.run(['pip', 'install', '-q', 'faiss-gpu'], capture_output=True).returncode != 0:
    subprocess.run(['pip', 'install', '-q', 'faiss-cpu'], capture_output=True)

os.chdir(f'{REPO}/BoT-SORT')
os.system('python setup.py develop -q 2>/dev/null')
os.chdir(f'{REPO}/deep-person-reid')
os.system('python setup.py develop -q 2>/dev/null')
os.chdir(REPO)
os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')

check = subprocess.run(
    'python -c "from thop import profile; from loguru import logger; '
    'from yolox.exp import get_exp; from tracker.bot_sort import BoTSORT; print(\"ALL OK\")"',
    shell=True, capture_output=True, text=True)
if 'ALL OK' in check.stdout:
    print('✓ All dependencies installed.')
else:
    print('✗ FAILED:', check.stderr.strip()[-400:])

---
## Step 2 — Check GPU

In [ ]:
r = subprocess.run(
    'python -c "import torch; print(\"CUDA:\", torch.cuda.is_available()); '
    'print(\"GPU:\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"None\")"',
    shell=True, capture_output=True, text=True)
print(r.stdout.strip())

if 'CUDA: False' in r.stdout:
    raise RuntimeError(
        '\n\n  ❌  NO GPU — go to Runtime → Change runtime type → T4 GPU\n'
        '  Then restart and re-run from Step 0.\n'
    )

---
## Step 3 — Download pretrained models

In [ ]:
os.system('pip install -q gdown')
M = f'{DRIVE}/models'

def get_model(local, on_drive, gdrive_id, name):
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.exists(local):
        print(f'{name}: already local')
    elif os.path.exists(on_drive):
        shutil.copy(on_drive, local)
        print(f'{name}: copied from Drive')
    else:
        print(f'{name}: downloading...')
        os.system(f'gdown "https://drive.google.com/uc?id={gdrive_id}" -O "{on_drive}"')
        shutil.copy(on_drive, local)
        print(f'{name}: done (cached to Drive)')

get_model(
    f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar',
    f'{M}/osnet_ms_m_c.pth.tar',
    '1IosIFlLiulGIjwW3H8uMCC3YvMyr9gZ2', 'OSNet (ReID)'
)
get_model(
    f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar',
    f'{M}/bytetrack_x_mot17.pth.tar',
    '1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', 'ByteTrack'
)

aic25_drive = f'{M}/ai_city_ckpt.pth.tar'
aic25_local = f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'
if os.path.exists(aic25_drive):
    shutil.copy(aic25_drive, aic25_local)
    print('AIC25 detector: copied from Drive ✓')
else:
    print('AIC25 detector: not found on Drive — ByteTrack base model will be used')

---
## Step 4 — Download Warehouse_017 from HuggingFace

> First time: downloads ~870 MB and saves to Drive.  
> Next sessions: loads from Drive in ~2 min — no re-download.

In [ ]:
from huggingface_hub import snapshot_download, login

drive_videos = f'{DRIVE}/datasets/{DATASET}/{SCENE}/videos'
local_data   = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
local_videos = f'{local_data}/videos'
os.makedirs(local_data, exist_ok=True)

# Download from HuggingFace if not already on Drive
if os.path.exists(drive_videos) and os.listdir(drive_videos):
    print(f'[CACHE HIT] Videos already on Drive ({len(os.listdir(drive_videos))} files) — skipping download.')
else:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('HF token loaded from Colab Secrets.')
    except Exception:
        import getpass
        hf_token = getpass.getpass('Paste HuggingFace token and press Enter: ')

    login(token=hf_token)
    print(f'Downloading {SCENE} videos (~870 MB)...')
    snapshot_download(
        repo_id='nvidia/PhysicalAI-SmartSpaces',
        repo_type='dataset',
        local_dir='/content/hf_tmp',
        allow_patterns=[
            f'MTMC_Tracking_2025/test/{SCENE}/videos/**',
            f'MTMC_Tracking_2025/test/{SCENE}/calibration.json',
        ]
    )
    hf_src = f'/content/hf_tmp/MTMC_Tracking_2025/test/{SCENE}'
    os.makedirs(drive_videos, exist_ok=True)
    for f in os.listdir(f'{hf_src}/videos'):
        shutil.copy(f'{hf_src}/videos/{f}', f'{drive_videos}/{f}')
    if os.path.exists(f'{hf_src}/calibration.json'):
        shutil.copy(f'{hf_src}/calibration.json',
                    f'{DRIVE}/datasets/{DATASET}/{SCENE}/calibration.json')
    shutil.rmtree('/content/hf_tmp', ignore_errors=True)
    print('Saved to Drive ✓')

# Copy calibration locally
cal_drive = f'{DRIVE}/datasets/{DATASET}/{SCENE}/calibration.json'
cal_local = f'{local_data}/calibration.json'
if os.path.exists(cal_drive) and not os.path.exists(cal_local):
    shutil.copy(cal_drive, cal_local)

# Local videos dir (flat mp4s — no symlink to Drive)
if os.path.islink(local_videos):
    os.unlink(local_videos)
os.makedirs(local_videos, exist_ok=True)
os.makedirs(f'{local_data}/depth_map', exist_ok=True)

mp4s = sorted(f for f in os.listdir(drive_videos) if f.endswith('.mp4'))
print(f'\n✓ {len(mp4s)} videos ready: {[os.path.splitext(f)[0] for f in mp4s]}')

---
## Step 5 — Link outputs to Drive
Detection / EmbedFeature / Tracking folders write directly to Drive.

In [ ]:
os.chdir(REPO)
for folder in ['Detection', 'EmbedFeature', 'Tracking']:
    drive_folder = f'{DRIVE}/outputs/{folder}'
    repo_folder  = f'{REPO}/{folder}'
    os.makedirs(drive_folder, exist_ok=True)
    if os.path.islink(repo_folder):
        pass
    elif os.path.isdir(repo_folder):
        shutil.move(repo_folder, drive_folder)
        os.symlink(drive_folder, repo_folder)
    else:
        os.symlink(drive_folder, repo_folder)
    print(f'{folder}/ → {drive_folder}')
print('\n✓ All outputs will save to Drive automatically.')

---
## Step 6 — Extract frames + Detection (per camera)
Heaviest step — ~10-20 min per camera on T4 GPU.  
Already-detected cameras are skipped automatically.

In [ ]:
import cv2
os.chdir(REPO)

if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
    ckpt     = 'BoT-SORT/ai_city_ckpt.pth.tar'
    exp_file = 'BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'
else:
    ckpt     = 'BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'
    exp_file = 'BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'
    print('⚠ Using ByteTrack base model (AIC25 checkpoint not found on Drive)')

print(f'Checkpoint : {ckpt}')
print(f'Cameras    : {CAMERAS}\n')

for i, cam in enumerate(CAMERAS):
    print(f'\n{"="*55}')
    print(f'  Camera {i+1}/{len(CAMERAS)}: {cam}')
    print(f'{"="*55}')

    det_txt = f'{REPO}/Detection/{SCENE}/{cam}.txt'
    if os.path.exists(det_txt):
        print(f'[SKIP] Detection already done for {cam}')
        continue

    # Extract frames from Drive mp4 to local SSD
    mp4 = f'{drive_videos}/{cam}.mp4'
    if not os.path.exists(mp4):
        print(f'[ERROR] {mp4} not found')
        continue

    frame_dir = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos/{cam}/Frame'
    if not (os.path.exists(frame_dir) and os.listdir(frame_dir)):
        print(f'[B] Extracting frames from {cam}.mp4...')
        os.makedirs(frame_dir, exist_ok=True)
        cap   = cv2.VideoCapture(mp4)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        n = 1
        ok, frm = cap.read()
        while ok:
            cv2.imwrite(f'{frame_dir}/{str(n).zfill(6)}.jpg', frm)
            ok, frm = cap.read()
            if n % 500 == 0:
                print(f'  {n}/{total} frames', flush=True)
            n += 1
        cap.release()
        print(f'[B] Extracted {n-1} frames')
    else:
        print(f'[B] Frames already extracted ({len(os.listdir(frame_dir))} files)')

    # Run detection
    print(f'[C] Running detection on {cam}...')
    cmd = (f'python BoT-SORT/tools/aic25_get_detection.py '
           f'--scene {SCENE} --dataset {DATASET} --camera {cam} '
           f'-f {exp_file} -c {ckpt} ./')
    result = subprocess.run(cmd, shell=True, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print('\n'.join(result.stdout.strip().splitlines()[-30:]))
    if result.returncode != 0:
        print(f'[ERROR] Detection failed for {cam}')
        break

    # Free local SSD
    shutil.rmtree(f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos/{cam}', ignore_errors=True)
    print(f'[FREE] Frames deleted — SSD free')

print('\n✓ Detection complete.')

---
## Step 7 — ReID Embeddings

In [ ]:
os.chdir(f'{REPO}/deep-person-reid')
ret = os.system(f'python torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../')
print('✓ ReID embeddings done' if ret == 0 else f'ERROR {ret}')

---
## Step 8 — Single-Camera Tracking → Camera.json

In [ ]:
os.chdir(REPO)
print(f'Tracking {len(CAMERAS)} cameras...')
for cam in CAMERAS:
    out = f'{REPO}/Tracking/Singlecamera/{SCENE}/{cam}/{cam}.json'
    if os.path.exists(out):
        print(f'[SKIP] {cam} — already tracked')
        continue
    print(f'--- {cam} ---')
    ret = os.system(f'python BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')
print('\n✓ Camera.json files ready.')

---
## Step 9 — Fix Single-Camera Results → fixed_Camera.json

In [ ]:
os.chdir(REPO)
for cam in CAMERAS:
    out = f'{REPO}/Tracking/Singlecamera/{SCENE}/{cam}/fixed_{cam}.json'
    if os.path.exists(out):
        print(f'[SKIP] {cam} — already fixed')
        continue
    print(f'--- {cam} ---')
    ret = os.system(f'python BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET} --nms')
    print('OK' if ret == 0 else f'ERROR {ret}')
print('\n✓ fixed_Camera.json files ready.')

---
## Step 10 — Results Summary + Drive paths

In [ ]:
import json
os.chdir(REPO)

print(f'{"Camera":<12} {"Raw tracklets":>14} {"Fixed tracklets":>16} {"Drive path"}')
print('-' * 90)

for cam in CAMERAS:
    raw_path   = f'Tracking/Singlecamera/{SCENE}/{cam}/{cam}.json'
    fixed_path = f'Tracking/Singlecamera/{SCENE}/{cam}/fixed_{cam}.json'

    raw_ids, fixed_ids = '-', '-'

    if os.path.exists(raw_path):
        with open(raw_path) as f:
            d = json.load(f)
        ids = {t.get('object sc id') for tracks in d.values() for t in tracks}
        raw_ids = str(len(ids))

    if os.path.exists(fixed_path):
        with open(fixed_path) as f:
            d = json.load(f)
        ids = {t.get('object sc id') for tracks in d.values() for t in tracks}
        fixed_ids = str(len(ids))

    drive_out = f'{DRIVE}/outputs/Tracking/Singlecamera/{SCENE}/{cam}/'
    exists_mark = '✓' if os.path.exists(fixed_path) else '✗'
    print(f'{cam:<12} {raw_ids:>14} {fixed_ids:>16}   {exists_mark} {drive_out}')

print(f'\nAll outputs saved to Drive under:')
print(f'  {DRIVE}/outputs/Tracking/Singlecamera/{SCENE}/')